# 05 — Legal Aid & Court Coverage Ratios: Jawa Barat (`32`) vs. Nusa Tenggara Barat (`52`)

**Project:** Justice for Rural Communities Observatory (JRC Observatory)  
**Data Tier:** `P1` (`ds02_pilot_courts_ntb_jabar.json`, `ds05_pilot_obh_ntb_jabar.json`, `kemendesa_p1_pilot_2025_indeks_desa.json`, `bps_p1_pilot_2014_2018_baseline.json`).  
**Caveats:**
1. BPHN OBH directory listing does not prove an active Posbakum contract at any specific court.
2. 8 court records have legacy BSDK addresses marked `pending_verification` and are excluded from geocoding.
3. Ratios are descriptive institutional coverage indicators, not measures of legal aid demand or quality.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve() if Path('../data').exists() else Path('.').resolve()
PUB = ROOT / 'data' / 'approved-public'

ds02 = pd.DataFrame(json.loads((PUB / 'ds02_pilot_courts_ntb_jabar.json').read_text(encoding='utf-8'))['records'])
ds05 = pd.DataFrame(json.loads((PUB / 'ds05_pilot_obh_ntb_jabar.json').read_text(encoding='utf-8'))['records'])
id25 = json.loads((PUB / 'kemendesa_p1_pilot_2025_indeks_desa.json').read_text(encoding='utf-8'))['provinces']
bps16 = json.loads((PUB / 'bps_p1_pilot_2014_2018_baseline.json').read_text(encoding='utf-8'))['tables']

kab_counts = {
    '32': int(bps16['Jumlah Kabupaten, 2016.csv']['JAWA BARAT'][0]) + int(bps16['Jumlah Kota, 2016.csv']['JAWA BARAT'][0]),
    '52': int(bps16['Jumlah Kabupaten, 2016.csv']['NUSA TENGGARA BARAT'][0]) + int(bps16['Jumlah Kota, 2016.csv']['NUSA TENGGARA BARAT'][0]),
}
desa_counts = {
    '32': id25['32_JAWA_BARAT']['villages_submitted'],
    '52': id25['52_NUSA_TENGGARA_BARAT']['villages_submitted'],
}

summary_rows = []
for code, name in [('32', 'Jawa Barat'), ('52', 'Nusa Tenggara Barat')]:
    c_sub = ds02[ds02['province_code'] == code]
    o_sub = ds05[ds05['area_id'].str.startswith(code)]
    pn_cnt = int((c_sub['institution_type'] == 'Pengadilan Negeri').sum())
    pa_cnt = int((c_sub['institution_type'] == 'Pengadilan Agama').sum())
    obh_cnt = len(o_sub)
    kab_kota = kab_counts[code]
    desa_n = desa_counts[code]
    summary_rows.append({
        'province': name,
        'kab_kota': kab_kota,
        'desa_indeks_2025': desa_n,
        'pn_count': pn_cnt,
        'pa_count': pa_cnt,
        'total_courts': len(c_sub),
        'obh_bphn_count': obh_cnt,
        'desa_per_court': round(desa_n / len(c_sub), 1),
        'desa_per_obh': round(desa_n / obh_cnt, 1),
        'obh_per_kab_kota': round(obh_cnt / kab_kota, 2),
    })

ratio_df = pd.DataFrame(summary_rows)
print(ratio_df.to_string(index=False))

In [ ]:
ds05['province_code'] = ds05['area_id'].str[:2]
grade_tbl = pd.crosstab(ds05['province_code'], ds05['accreditation_grade_displayed'], margins=True)
print('\nBPHN OBH Accreditation Grades Displayed (32=Jawa Barat, 52=NTB):')
print(grade_tbl)